In [1]:
import sys
import os

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nWorking directory:")
print(os.getcwd())

try:
    import qiskit
    print("\nQiskit version:")
    print(qiskit.__version__)
except Exception as e:
    print("\nQiskit:")
    print("ERROR:", e)

try:
    import qiskit_ibm_runtime
    print("\nQiskit IBM Runtime version:")
    print(qiskit_ibm_runtime.__version__)
except Exception as e:
    print("\nQiskit IBM Runtime:")
    print("ERROR:", e)

Python version:
3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]

Python executable:
C:\Users\heart\anaconda3\python.exe

Working directory:
C:\Users\heart\quantum_6g_ai

Qiskit version:
2.5.2

Qiskit IBM Runtime version:
0.49.0


In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform"
)

backend_names = [
    "ibm_fez",
    "ibm_marrakesh",
    "ibm_kingston"
]

for name in backend_names:
    backend = service.backend(name)

    print("=" * 60)
    print(f"Backend       : {backend.name}")
    print(f"Qubits        : {backend.num_qubits}")

    try:
        print(f"Operational   : {backend.status().operational}")
    except Exception as e:
        print(f"Operational   : ERROR - {e}")

    try:
        print(f"Pending jobs  : {backend.status().pending_jobs}")
    except Exception as e:
        print(f"Pending jobs  : ERROR - {e}")

qiskit_runtime_service.__init__:WARNING:2026-09-18 11:53:35,292: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-09-18 11:53:35,298: Using instance: open-instance, plan: open


Backend       : ibm_fez
Qubits        : 156
Operational   : True


qiskit_runtime_service.backends:WARNING:2026-09-18 11:53:39,596: Using instance: open-instance, plan: open


Pending jobs  : 97
Backend       : ibm_marrakesh
Qubits        : 156
Operational   : True


qiskit_runtime_service.backends:WARNING:2026-09-18 11:53:43,362: Using instance: open-instance, plan: open


Pending jobs  : 1
Backend       : ibm_kingston
Qubits        : 156
Operational   : True
Pending jobs  : 1


In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import SamplerV2

# Select IBM Quantum hardware
backend = service.backend("ibm_marrakesh")

# Simple 2-qubit test circuit
qc = QuantumCircuit(2)

qc.h(0)
qc.cx(0, 1)

qc.measure_all()

print(qc)
print()
print("Backend:", backend.name)
print("Qubits :", backend.num_qubits)

qiskit_runtime_service.backends:WARNING:2026-09-18 11:54:35,636: Using instance: open-instance, plan: open


        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

Backend: ibm_marrakesh
Qubits : 156


In [4]:
from qiskit_ibm_runtime import SamplerV2

sampler = SamplerV2(mode=backend)

job = sampler.run([qc], shots=1024)

print("Job submitted successfully.")
print("Job ID:", job.job_id())

IBMInputValueError: 'The instruction h on qubits (0,) is not supported by the target system. Circuits that do not match the target hardware definition are no longer supported after March 4, 2024. See the transpilation documentation (https://quantum.cloud.ibm.com/docs/guides/transpile) for instructions to transform circuits and the primitive examples (https://quantum.cloud.ibm.com/docs/guides/primitives-examples) to see this coupled with operator transformations.'

In [5]:
from qiskit.transpiler import generate_preset_pass_manager

# Create an IBM hardware-specific transpiler
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
    seed_transpiler=42
)

# Convert abstract circuit -> ISA circuit
isa_qc = pm.run(qc)

print("Original circuit:")
print(qc)

print("\n" + "=" * 60)
print("ISA circuit for:", backend.name)
print("=" * 60)
print(isa_qc)

print("\nISA operations:")
print(isa_qc.count_ops())

Original circuit:
        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

ISA circuit for: ibm_marrakesh
global phase: 3π/4
         ┌─────────┐┌────┐┌─────────┐                                ░ ┌─┐   
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■──────────────────────────────░─┤M├───
         ├─────────┤├────┤├─────────┤ │ ┌─────────┐┌────┐┌─────────┐ ░ └╥┘┌─┐
q_1 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─░──╫─┤M├
         └─────────┘└────┘└─────────┘   └─────────┘└────┘└─────────┘ ░  ║ └╥┘
 meas: 2/═══════════════════════════════════════════════════════════════╩══╩═
                                                                        0  1 

ISA operations:
OrderedDict([('rz', 6), ('sx', 3), ('measure', 2), ('cz', 1), ('barrier', 1)])


In [6]:
from qiskit_ibm_runtime import SamplerV2

sampler = SamplerV2(mode=backend)

job = sampler.run(
    [isa_qc],
    shots=1024
)

print("Job submitted successfully.")
print("Job ID:", job.job_id())
print("Status:", job.status())

Job submitted successfully.
Job ID: damcmu8pqrnc7397b1m0
Status: QUEUED


In [7]:
result = job.result()

print("Hardware job completed.")
print(result)

Hardware job completed.
PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-09-18 05:26:43', stop='2026-09-18 05:26:45', size=1024>)])}, 'version': 2})


In [8]:
pub_result = result[0]

print(pub_result.data)

DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>))


In [9]:
print("Job ID:", job.job_id())
print("Status:", job.status())

Job ID: damcmu8pqrnc7397b1m0
Status: DONE


In [10]:
result = job.result()

print("Hardware execution completed.")
print(result)

Hardware execution completed.
PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-09-18 05:26:43', stop='2026-09-18 05:26:45', size=1024>)])}, 'version': 2})


In [11]:
pub_result = result[0]

print(pub_result.data)

DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>))


In [12]:
# Extract measurement counts
counts = result[0].data.meas.get_counts()

print("Measurement counts:")
print("=" * 40)

for state, count in sorted(counts.items()):
    print(f"{state}: {count}")

print("\nTotal shots:", sum(counts.values()))

Measurement counts:
00: 499
01: 10
10: 6
11: 509

Total shots: 1024


In [13]:
# Convert counts to probabilities

total_shots = sum(counts.values())

probabilities = {
    state: count / total_shots
    for state, count in counts.items()
}

print("Measurement probabilities:")
print("=" * 40)

for state, probability in sorted(probabilities.items()):
    print(f"{state}: {probability:.4f} ({probability * 100:.2f}%)")

Measurement probabilities:
00: 0.4873 (48.73%)
01: 0.0098 (0.98%)
10: 0.0059 (0.59%)
11: 0.4971 (49.71%)


In [14]:
import csv
import os
from datetime import datetime

counts = result[0].data.meas.get_counts()

total_shots = sum(counts.values())

p00 = counts.get("00", 0) / total_shots
p01 = counts.get("01", 0) / total_shots
p10 = counts.get("10", 0) / total_shots
p11 = counts.get("11", 0) / total_shots

bell_fidelity_proxy = p00 + p11
error_probability = p01 + p10

record = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "experiment": "Bell-state hardware validation",
    "backend": backend.name,
    "job_id": job.job_id(),
    "shots": total_shots,
    "count_00": counts.get("00", 0),
    "count_01": counts.get("01", 0),
    "count_10": counts.get("10", 0),
    "count_11": counts.get("11", 0),
    "p00": p00,
    "p01": p01,
    "p10": p10,
    "p11": p11,
    "bell_fidelity_proxy": bell_fidelity_proxy,
    "error_probability": error_probability,
}

output_file = "results/hardware_runs.csv"

file_exists = os.path.exists(output_file)

with open(output_file, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=record.keys())

    if not file_exists:
        writer.writeheader()

    writer.writerow(record)

print("Hardware experiment saved:")
print(output_file)
print()
print(record)

Hardware experiment saved:
results/hardware_runs.csv

{'timestamp': '2026-09-18T12:02:29', 'experiment': 'Bell-state hardware validation', 'backend': 'ibm_marrakesh', 'job_id': 'damcmu8pqrnc7397b1m0', 'shots': 1024, 'count_00': 499, 'count_01': 10, 'count_10': 6, 'count_11': 509, 'p00': 0.4873046875, 'p01': 0.009765625, 'p10': 0.005859375, 'p11': 0.4970703125, 'bell_fidelity_proxy': 0.984375, 'error_probability': 0.015625}
